# HyperPyYAML Tutorial

An essential aspect of any deep learning pipeline is the definition of hyperparameters and other metadata. These hyperparameters, in conjunction with deep learning algorithms, govern various aspects of the pipeline, including model architecture, training, and decoding.

In Real-Time-Speech-Separation-Model-Toolkit, we emphasize a clear distinction between hyperparameters and learning algorithms in the structure of our toolkit. To achieve this, we separate our recipes into two primary files: `train.py` and `train.yaml`.

The `train.yaml` file follows a format developed by Real-Time-Speech-Separation-Model-Toolkit, known as "HyperPyYAML." We chose to extend YAML due to its highly readable nature for data serialization. By building upon this already user-friendly format, we have created an extended definition of hyperparameters, ensuring that our experimental code remains concise and easily readable.

Here's a brief example using PyTorch code to illustrate the use of HyperPyYAML. It's important to note that PyTorch is not a requirement for utilizing HyperPyYAML:

In [ ]:
%%capture
!pip install torch
!pip install hyperpyyaml

In [ ]:
import torch
from hyperpyyaml import load_hyperpyyaml

example_hyperparams = """
base_channels: 32
kernel_size: 11
padding: !ref <kernel_size> // 2

layer1: !new:torch.nn.Conv1d
  in_channels: 1
  out_channels: !ref <base_channels>
  kernel_size: !ref <kernel_size>
  padding: !ref <padding>

layer2: !new:torch.nn.Conv1d
  in_channels: !ref <base_channels>
  out_channels: !ref <base_channels> * 2
  kernel_size: !ref <kernel_size>
  padding: !ref <padding>

layer3: !new:torch.nn.Conv1d
  in_channels: !ref <base_channels> * 2
  out_channels: 1
  kernel_size: !ref <kernel_size>
  padding: !ref <padding>

model: !new:torch.nn.Sequential
  - !ref <layer1>
  - !new:torch.nn.LeakyReLU
  - !ref <layer2>
  - !new:torch.nn.LeakyReLU
  - !ref <layer3>
"""

# Create model directly by loading YAML
loaded_hparams = load_hyperpyyaml(example_hyperparams)
model = loaded_hparams["model"]

# Transform a 2-second audio clip
input_audio = torch.rand(1, 1, 32000)
transformed_audio = model(input_audio)
print(transformed_audio.shape)

# Try a different hyperparameter value by overriding the padding value
loaded_hparams = load_hyperpyyaml(example_hyperparams, {"padding": 0})
model = loaded_hparams["model"]
transformed_audio = model(input_audio)
print(transformed_audio.shape)

As this example shows, HyperPyYAML allows for complex hyperparameter definitions with compositions. In addition, any value can be overridden for hyperparameter tuning. To grasp how all of this works, let's first briefly look at the basics of YAML.

## Basic YAML syntax

Enough prelude: let's talk YAML! Here's a brief example of a yaml snippet and what it would look like once loaded to python:

In [ ]:
import yaml
yaml_string = """
foo: 1
bar:
  - item1
  - item2
baz:
  item1: 3.4
  item2: True
"""
yaml.safe_load(yaml_string)

## HyperPyYAML Extensions

Real-Time-Speech-Separation-Model-Toolkit extends standard YAML with several powerful features:

### 1. Object Creation with `!new:`

The `!new:` tag allows you to create objects directly from YAML:

In [ ]:
yaml_string = """
# Create a simple linear layer
linear_layer: !new:torch.nn.Linear
  in_features: 10
  out_features: 5
"""

hparams = load_hyperpyyaml(yaml_string)
layer = hparams["linear_layer"]
print(f"Created layer: {layer}")
print(f"Layer type: {type(layer)}")

### 2. References with `!ref:`

The `!ref:` tag allows you to reference other values in the YAML file:

In [ ]:
yaml_string = """
input_size: 100
hidden_size: 50
output_size: 10

model: !new:torch.nn.Sequential
  - !new:torch.nn.Linear
    in_features: !ref <input_size>
    out_features: !ref <hidden_size>
  - !new:torch.nn.ReLU
  - !new:torch.nn.Linear
    in_features: !ref <hidden_size>
    out_features: !ref <output_size>
"""

hparams = load_hyperpyyaml(yaml_string)
model = hparams["model"]
print(f"Model: {model}")

# Test with different input size
hparams_override = load_hyperpyyaml(yaml_string, {"input_size": 200})
model_override = hparams_override["model"]
print(f"Model with override: {model_override}")

### 3. Function Calls with `!name:`

The `!name:` tag allows you to call functions:

In [ ]:
yaml_string = """
# Create activation functions
activation1: !name:torch.nn.ReLU []
activation2: !name:torch.nn.Tanh []
activation3: !name:torch.nn.Sigmoid []
"""

hparams = load_hyperpyyaml(yaml_string)
relu = hparams["activation1"]
tanh = hparams["activation2"]
sigmoid = hparams["activation3"]

print(f"ReLU: {relu}")
print(f"Tanh: {tanh}")
print(f"Sigmoid: {sigmoid}")

### 4. Mathematical Expressions

HyperPyYAML supports mathematical expressions in references:

In [ ]:
yaml_string = """
base_size: 64
growth_factor: 2

layer_sizes:
  layer1: !ref <base_size>
  layer2: !ref <base_size> * <growth_factor>
  layer3: !ref <base_size> * <growth_factor> ^ 2
  dropout: !ref <base_size> / 100.0
"""

hparams = load_hyperpyyaml(yaml_string)
sizes = hparams["layer_sizes"]
print(f"Layer 1 size: {sizes['layer1']}")
print(f"Layer 2 size: {sizes['layer2']}")
print(f"Layer 3 size: {sizes['layer3']}")
print(f"Dropout rate: {sizes['dropout']}")

## Advanced Features

### Nested References and Complex Objects

In [ ]:
yaml_string = """
# Define a complex neural network
features:
  conv: !new:torch.nn.Conv1d
    in_channels: 1
    out_channels: 32
    kernel_size: 3
    padding: 1
  activation: !name:torch.nn.ReLU []
  pool: !new:torch.nn.MaxPool1d
    kernel_size: 2

classifier:
  linear1: !new:torch.nn.Linear
    in_features: !ref <features>['conv'].out_channels * 16000 // 2
    out_features: 128
  activation: !name:torch.nn.ReLU []
  linear2: !new:torch.nn.Linear
    in_features: 128
    out_features: 10

model: !new:torch.nn.Sequential
  - !ref <features>['conv']
  - !ref <features>['activation']
  - !ref <features>['pool']
  - !name:torch.nn.Flatten [start_dim=1]
  - !ref <classifier>['linear1']
  - !ref <classifier>['activation']
  - !ref <classifier>['linear2']
"""

hparams = load_hyperpyyaml(yaml_string)
model = hparams["model"]
print(f"Complex model: {model}")

# Test the model
test_input = torch.randn(1, 1, 16000)
output = model(test_input)
print(f"Output shape: {output.shape}")

### Conditional Logic and Overrides

In [ ]:
yaml_string = """
model_type: "simple"
hidden_size: 128

# Define different model architectures
simple_model: !new:torch.nn.Linear
  in_features: 16000
  out_features: !ref <hidden_size>

complex_model: !new:torch.nn.Sequential
  - !new:torch.nn.Linear
    in_features: 16000
    out_features: !ref <hidden_size>
  - !name:torch.nn.ReLU []
  - !new:torch.nn.Linear
    in_features: !ref <hidden_size>
    out_features: !ref <hidden_size>
  - !name:torch.nn.ReLU []
  - !new:torch.nn.Linear
    in_features: !ref <hidden_size>
    out_features: 10

# Select model based on type
model: !ref <simple_model> if <model_type> == "simple" else <complex_model>
"""

# Load simple model
hparams_simple = load_hyperpyyaml(yaml_string)
simple_model = hparams_simple["model"]
print(f"Simple model: {simple_model}")

# Load complex model with override
hparams_complex = load_hyperpyyaml(yaml_string, {"model_type": "complex"})
complex_model = hparams_complex["model"]
print(f"Complex model: {complex_model}")

## Integration with Real-Time-Speech-Separation-Model-Toolkit

HyperPyYAML integrates seamlessly with the Brain class:

In [ ]:
yaml_string = """
# Model definition
model: !new:torch.nn.Sequential
  - !new:torch.nn.Linear
    in_features: 16000
    out_features: 512
  - !name:torch.nn.ReLU []
  - !new:torch.nn.Linear
    in_features: 512
    out_features: 128
  - !name:torch.nn.ReLU []
  - !new:torch.nn.Linear
    in_features: 128
    out_features: 10

# Optimizer definition
optimizer: !name:torch.optim.Adam
  lr: 0.001
  weight_decay: 0.0001

# Training parameters
batch_size: 32
epochs: 100
"""

class MockBrain:
    def __init__(self, hparams):
        self.hparams = hparams
        self.modules = hparams
        self.optimizer = hparams["optimizer"](self.modules["model"].parameters())
    
    def compute_forward(self, batch):
        return self.modules["model"](batch)
    
    def compute_objectives(self, predictions, targets):
        return torch.nn.functional.mse_loss(predictions, targets)

# Load hyperparameters
hparams = load_hyperpyyaml(yaml_string)

# Create brain instance
brain = MockBrain(hparams)

print(f"Model: {brain.modules['model']}")
print(f"Optimizer: {brain.optimizer}")
print(f"Batch size: {brain.hparams['batch_size']}")
print(f"Epochs: {brain.hparams['epochs']}")

## Best Practices

### 1. Organize Your YAML Files
- Use clear, descriptive names for parameters
- Group related parameters together
- Use comments to explain complex configurations

### 2. Use References Effectively
- Define base values once and reference them multiple times
- Use mathematical expressions to derive related values
- Avoid hardcoding repeated values

### 3. Modular Design
- Define reusable components
- Use conditional logic for different configurations
- Keep model and training parameters separate

## Summary

HyperPyYAML in Real-Time-Speech-Separation-Model-Toolkit provides:

1. **Clean Configuration**: Separate hyperparameters from code
2. **Flexible References**: Link parameters together
3. **Dynamic Object Creation**: Instantiate objects from YAML
4. **Easy Override**: Change parameters without editing files
5. **Mathematical Expressions**: Compute derived values
6. **Integration**: Works seamlessly with training pipelines

By using HyperPyYAML effectively, you can create maintainable and configurable experiments that are easy to reproduce and modify.